# Regional Weather Preparation

This notebook:

1. loads census state centroids and population weights;
2. selects states for an EIA storage region;
3. downloads daily mean temperatures from Open-Meteo;
4. calculates HDD/CDD at state level;
5. aggregates to a population-weighted regional weather index;
6. exports processed datasets for modeling.

In [ ]:
from pathlib import Path

import pandas as pd

from gas_forecast.data.export import save_versioned_parquet
from gas_forecast.data.regions import region_slug, region_states
from gas_forecast.data.weather import (
    aggregate_population_weighted_weather,
    calculate_hdd_cdd,
    fetch_all_state_temperatures,
    load_census_state_locations,
    prepare_weather_model_data,
    select_weather_locations,
    validate_state_daily_weather,
)

In [ ]:
REGION = "R48"  # R48, R01, R02, R03, R04, R05

PROCESSED_DIR = Path("../data/processed")
RAW_DIR = Path("../data/raw")
WEATHER_CACHE_DIR = RAW_DIR / "weather_cache"

REGION_SLUG = region_slug(REGION)
STORAGE_PATH = PROCESSED_DIR / f"{REGION_SLUG}_weekly_storage_latest.parquet"

In [ ]:
storage = pd.read_parquet(STORAGE_PATH)

START_DATE = storage["date"].min().strftime("%Y-%m-%d")
END_DATE = storage["date"].max().strftime("%Y-%m-%d")

In [ ]:
locations = load_census_state_locations()
locations = select_weather_locations(locations, REGION)

locations.head()

In [ ]:
state_daily_weather = fetch_all_state_temperatures(
    locations=locations,
    start_date=START_DATE,
    end_date=END_DATE,
    location_batch_size=1,
    date_freq="YS",
    pause_seconds=3.0,
    cache_dir=WEATHER_CACHE_DIR,
)

In [ ]:
validate_state_daily_weather(
    state_daily_weather,
    expected_states=region_states(REGION),
)

state_daily_weather.head()

In [ ]:
save_versioned_parquet(
    state_daily_weather,
    PROCESSED_DIR,
    f"{REGION_SLUG}_state_daily_weather",
    save_latest=True,
)

In [ ]:
state_degrees = calculate_hdd_cdd(state_daily_weather)
regional_weather = aggregate_population_weighted_weather(state_degrees)

regional_weather.head()

In [ ]:
regional_weather_model = prepare_weather_model_data(regional_weather, REGION)

save_versioned_parquet(
    regional_weather_model,
    PROCESSED_DIR,
    f"{REGION_SLUG}_daily_weather",
    save_latest=True,
)